# Historical Replay Deployment Prototype

This notebook is a historical replay demo for the NBA statline forecasting project. It is **not** a live same-day prediction app. The goal is to show how the trained project models can be used on a real held-out game context from the `2024-25` evaluation season.

The replay uses only the already-engineered pregame features stored in the project dataset. It does **not** parse live injuries or real-time inputs yet.

## Demo Workflow

1. Load the existing ML-ready player-game dataset.
2. Reuse the feature definitions and model families from `03_model_training.ipynb`.
3. Select the best configuration for each target from `outputs/results/model_results.csv` when available.
4. Retrain those selected models on the `2023-24` training season only.
5. Replay a real held-out `2024-25` game row for a chosen player and compare predicted vs actual points, rebounds, and assists.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False
    XGBRegressor = None

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 120)

ROOT = Path.cwd()
DATA_PATH = ROOT / 'data' / 'final' / 'ml_ready_player_games.parquet'
MODEL_RESULTS_PATH = ROOT / 'outputs' / 'results' / 'model_results.csv'

TRAIN_SEASON = '2023-24'
EVAL_SEASON = '2024-25'

print(f'Project root: {ROOT}')
print(f'Dataset exists: {DATA_PATH.exists()}')
print(f'Model results exists: {MODEL_RESULTS_PATH.exists()}')
print(f'XGBoost available: {XGBOOST_AVAILABLE}')

Project root: c:\Users\aruls\anaconda_projects\analytics\NBACapstone-main\ipynb
Dataset exists: True
Model results exists: True
XGBoost available: True


## Load The Existing ML-Ready Dataset

In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing dataset: {DATA_PATH}')

df = pd.read_parquet(DATA_PATH).copy()
df['game_date'] = pd.to_datetime(df['game_date'])

train_df = df[df['season'] == TRAIN_SEASON].copy()
eval_df = df[df['season'] == EVAL_SEASON].copy()

summary_df = pd.DataFrame([
    {'split': 'train', 'season': TRAIN_SEASON, 'rows': len(train_df), 'players': train_df['player_name'].nunique(), 'games': train_df['game_id'].nunique()},
    {'split': 'eval', 'season': EVAL_SEASON, 'rows': len(eval_df), 'players': eval_df['player_name'].nunique(), 'games': eval_df['game_id'].nunique()},
])

display(summary_df.style.hide(axis='index').format({'rows': '{:,.0f}', 'players': '{:,.0f}', 'games': '{:,.0f}'}))

split,season,rows,players,games
train,2023-24,"26,401",572,"1,230"
eval,2024-25,"26,306",569,"1,230"


## Reuse The Project Feature Sets And Model Families

This section matches the naming and feature-group logic already used in `03_model_training.ipynb`. Optional columns are handled defensively: if a planned column is missing from the current dataset, it is dropped from that replay configuration rather than invented.

In [3]:
TARGETS = {
    'target_pts': {'BaselineA': ['season_avg_pts'], 'BaselineB': ['l5_avg_pts']},
    'target_reb': {'BaselineA': ['season_avg_reb'], 'BaselineB': ['l5_avg_reb']},
    'target_ast': {'BaselineA': ['season_avg_ast'], 'BaselineB': ['l5_avg_ast']},
}

TARGET_LABELS = {
    'target_pts': 'Points',
    'target_reb': 'Rebounds',
    'target_ast': 'Assists',
}

FALLBACK_CONFIGS = {
    'target_pts': {'model_version': 'V2', 'model_family': 'RandomForestRegressor'},
    'target_reb': {'model_version': 'V2', 'model_family': 'RandomForestRegressor'},
    'target_ast': {'model_version': 'V2', 'model_family': 'RandomForestRegressor'},
}

def get_feature_sets(target):
    v0 = [
        'season_avg_pts', 'season_avg_reb', 'season_avg_ast', 'season_avg_min', 'season_avg_fga', 'season_avg_fg3a', 'season_avg_fta',
        'l5_avg_pts', 'l5_avg_reb', 'l5_avg_ast', 'l5_avg_min', 'l5_avg_fga', 'l5_avg_fg3a', 'l5_avg_fta',
        'l10_avg_pts', 'l10_avg_reb', 'l10_avg_ast', 'l10_avg_min', 'l10_avg_fga', 'l10_avg_fg3a', 'l10_avg_fta',
        'opponent_team', 'home_away'
    ]
    availability = [
        'teammates_out_count', 'missing_pts_l5', 'missing_reb_l5', 'missing_ast_l5', 'missing_min_l5',
        'missing_pts_l10', 'missing_reb_l10', 'missing_ast_l10', 'missing_min_l10'
    ]
    usage_context = ['season_avg_usg', 'l5_avg_usg', 'l10_avg_usg', 'missing_usg_l5', 'missing_usg_l10', 'opponent_pace', 'opponent_def_rating']
    return {
        'BaselineA': TARGETS[target]['BaselineA'],
        'BaselineB': TARGETS[target]['BaselineB'],
        'V0': v0,
        'V1': v0 + availability,
        'V2': v0 + availability + usage_context,
    }

def available_feature_cols(target, model_version, dataset_columns):
    feature_sets = get_feature_sets(target)
    if model_version not in feature_sets:
        raise KeyError(f'Unknown model_version={model_version} for {target}')
    return [col for col in feature_sets[model_version] if col in dataset_columns]

def make_pipeline(feature_cols, model_family):
    categorical = [c for c in feature_cols if c in ['opponent_team', 'home_away']]
    numeric = [c for c in feature_cols if c not in categorical]
    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical),
    ])
    if model_family == 'LinearRegression':
        model = LinearRegression()
    elif model_family == 'RandomForestRegressor':
        model = RandomForestRegressor(n_estimators=200, min_samples_leaf=2, random_state=42, n_jobs=-1)
    elif model_family == 'XGBRegressor':
        if not XGBOOST_AVAILABLE:
            raise ImportError('XGBRegressor was requested but xgboost is not available in this environment.')
        model = XGBRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='reg:squarederror',
            random_state=42,
            n_jobs=4,
        )
    else:
        raise ValueError(model_family)
    return Pipeline([('preprocessor', preprocessor), ('model', model)])

def load_best_configs(model_results_path, dataset_columns):
    if model_results_path.exists():
        results = pd.read_csv(model_results_path)
        needed = {'target', 'model_version', 'model_family', 'mae'}
        if needed.issubset(results.columns):
            results = results[results['target'].isin(TARGETS)].copy()
            if not XGBOOST_AVAILABLE:
                results = results[results['model_family'] != 'XGBRegressor'].copy()
            if not results.empty:
                best = results.sort_values(['target', 'mae']).groupby('target', as_index=False).first()
                configs = []
                for _, row in best.iterrows():
                    feature_cols = available_feature_cols(row['target'], row['model_version'], dataset_columns)
                    configs.append({
                        'target': row['target'],
                        'target_label': TARGET_LABELS[row['target']],
                        'selection_source': 'model_results.csv',
                        'model_version': row['model_version'],
                        'model_family': row['model_family'],
                        'mae': row['mae'],
                        'feature_cols': feature_cols,
                        'feature_count': len(feature_cols),
                    })
                configs_df = pd.DataFrame(configs)
                if set(configs_df['target']) == set(TARGETS):
                    return configs_df.sort_values('target').reset_index(drop=True)

    fallback_rows = []
    for target, cfg in FALLBACK_CONFIGS.items():
        feature_cols = available_feature_cols(target, cfg['model_version'], dataset_columns)
        fallback_rows.append({
            'target': target,
            'target_label': TARGET_LABELS[target],
            'selection_source': 'fallback_default',
            'model_version': cfg['model_version'],
            'model_family': cfg['model_family'],
            'mae': pd.NA,
            'feature_cols': feature_cols,
            'feature_count': len(feature_cols),
        })
    return pd.DataFrame(fallback_rows).sort_values('target').reset_index(drop=True)

selected_configs = load_best_configs(MODEL_RESULTS_PATH, set(df.columns))
selected_configs_display = selected_configs.copy()
selected_configs_display['feature_cols'] = selected_configs_display['feature_cols'].apply(lambda cols: ', '.join(cols))
display(selected_configs_display.style.hide(axis='index'))

target,target_label,selection_source,model_version,model_family,mae,feature_cols,feature_count
target_ast,Assists,model_results.csv,V2,LinearRegression,1.348317,"season_avg_pts, season_avg_reb, season_avg_ast, season_avg_min, season_avg_fga, season_avg_fg3a, season_avg_fta, l5_avg_pts, l5_avg_reb, l5_avg_ast, l5_avg_min, l5_avg_fga, l5_avg_fg3a, l5_avg_fta, l10_avg_pts, l10_avg_reb, l10_avg_ast, l10_avg_min, l10_avg_fga, l10_avg_fg3a, l10_avg_fta, opponent_team, home_away, teammates_out_count, missing_pts_l5, missing_reb_l5, missing_ast_l5, missing_min_l5, missing_pts_l10, missing_reb_l10, missing_ast_l10, missing_min_l10, season_avg_usg, l5_avg_usg, l10_avg_usg, missing_usg_l5, missing_usg_l10, opponent_pace, opponent_def_rating",39
target_pts,Points,model_results.csv,V2,XGBRegressor,4.610939,"season_avg_pts, season_avg_reb, season_avg_ast, season_avg_min, season_avg_fga, season_avg_fg3a, season_avg_fta, l5_avg_pts, l5_avg_reb, l5_avg_ast, l5_avg_min, l5_avg_fga, l5_avg_fg3a, l5_avg_fta, l10_avg_pts, l10_avg_reb, l10_avg_ast, l10_avg_min, l10_avg_fga, l10_avg_fg3a, l10_avg_fta, opponent_team, home_away, teammates_out_count, missing_pts_l5, missing_reb_l5, missing_ast_l5, missing_min_l5, missing_pts_l10, missing_reb_l10, missing_ast_l10, missing_min_l10, season_avg_usg, l5_avg_usg, l10_avg_usg, missing_usg_l5, missing_usg_l10, opponent_pace, opponent_def_rating",39
target_reb,Rebounds,model_results.csv,V0,LinearRegression,1.942623,"season_avg_pts, season_avg_reb, season_avg_ast, season_avg_min, season_avg_fga, season_avg_fg3a, season_avg_fta, l5_avg_pts, l5_avg_reb, l5_avg_ast, l5_avg_min, l5_avg_fga, l5_avg_fg3a, l5_avg_fta, l10_avg_pts, l10_avg_reb, l10_avg_ast, l10_avg_min, l10_avg_fga, l10_avg_fg3a, l10_avg_fta, opponent_team, home_away",23


## Train The Selected Historical Replay Models

Each target gets its own global model. Training uses only the `2023-24` season. The replay season `2024-25` stays held out until prediction time.

In [4]:
def train_selected_models(train_data, configs_df):
    trained = {}
    training_summary = []
    for _, config in configs_df.iterrows():
        target = config['target']
        feature_cols = list(config['feature_cols'])
        model_version = config['model_version']
        model_family = config['model_family']
        print(f'Training {target} with {model_version} + {model_family} on {len(train_data):,} rows...')
        pipeline = make_pipeline(feature_cols, model_family)
        pipeline.fit(train_data[feature_cols], train_data[target])
        trained[target] = {
            'pipeline': pipeline,
            'target': target,
            'target_label': config['target_label'],
            'model_version': model_version,
            'model_family': model_family,
            'feature_cols': feature_cols,
        }
        training_summary.append({
            'target': target,
            'target_label': config['target_label'],
            'model_version': model_version,
            'model_family': model_family,
            'feature_count': len(feature_cols),
        })
    return trained, pd.DataFrame(training_summary)

trained_models, training_summary_df = train_selected_models(train_df, selected_configs)
display(training_summary_df.style.hide(axis='index'))

Training target_ast with V2 + LinearRegression on 26,401 rows...
Training target_pts with V2 + XGBRegressor on 26,401 rows...
Training target_reb with V0 + LinearRegression on 26,401 rows...


target,target_label,model_version,model_family,feature_count
target_ast,Assists,V2,LinearRegression,39
target_pts,Points,V2,XGBRegressor,39
target_reb,Rebounds,V0,LinearRegression,23


## Replay Helpers

The helper functions below do two things:

- `list_player_games(player_name)` shows available held-out `2024-25` games for a player.
- `historical_replay(player_name, game_date)` replays one real held-out game row using only the pregame features already engineered into the dataset.

In [5]:
def _display_styled_table(title, frame, precision_map=None):
    display(Markdown(f'### {title}'))
    styler = frame.style.hide(axis='index')
    if precision_map:
        styler = styler.format(precision_map)
    display(styler)

def list_player_games(player_name, n=20):
    games = eval_df[eval_df['player_name'].str.lower() == player_name.lower()].copy()
    if games.empty:
        raise ValueError(f'No held-out {EVAL_SEASON} games found for player_name={player_name!r}')
    cols = ['game_date', 'team_abbr', 'opponent_team', 'home_away', 'target_pts', 'target_reb', 'target_ast']
    games = games.sort_values('game_date')[cols].head(n).copy()
    games['game_date'] = games['game_date'].dt.strftime('%Y-%m-%d')
    return games.rename(columns={
        'game_date': 'game_date',
        'team_abbr': 'team',
        'opponent_team': 'opponent',
        'home_away': 'home_away',
        'target_pts': 'actual_points',
        'target_reb': 'actual_rebounds',
        'target_ast': 'actual_assists',
    })

def historical_replay(player_name, game_date):
    game_date = pd.to_datetime(game_date)
    row = eval_df[(eval_df['player_name'].str.lower() == player_name.lower()) & (eval_df['game_date'] == game_date)].copy()
    if row.empty:
        raise ValueError(f'No held-out row found for player_name={player_name!r} and game_date={game_date.date()}')
    if len(row) > 1:
        raise ValueError('Expected exactly one player-game row, but found duplicates.')
    row = row.iloc[[0]].copy()

    context_table = pd.DataFrame([{
        'player_name': row.iloc[0]['player_name'],
        'game_date': row.iloc[0]['game_date'].strftime('%Y-%m-%d'),
        'team': row.iloc[0]['team_abbr'],
        'opponent': row.iloc[0]['opponent_team'],
        'home_away': row.iloc[0]['home_away'],
        'season': row.iloc[0]['season'],
        'game_id': row.iloc[0]['game_id'],
    }])

    replay_rows = []
    feature_usage_rows = []
    for target, bundle in trained_models.items():
        feature_cols = bundle['feature_cols']
        prediction = float(bundle['pipeline'].predict(row[feature_cols])[0])
        actual = float(row.iloc[0][target])
        abs_error = abs(prediction - actual)
        replay_rows.append({
            'target': bundle['target_label'],
            'model_version': bundle['model_version'],
            'model_family': bundle['model_family'],
            'prediction': prediction,
            'actual': actual,
            'absolute_error': abs_error,
        })
        for feature in feature_cols:
            feature_usage_rows.append({
                'feature': feature,
                'value': row.iloc[0][feature],
                'used_for_target': bundle['target_label'],
                'model_version': bundle['model_version'],
                'model_family': bundle['model_family'],
            })

    feature_usage = pd.DataFrame(feature_usage_rows)
    feature_table = (
        feature_usage.groupby('feature', as_index=False)
        .agg(
            value=('value', 'first'),
            used_for_targets=('used_for_target', lambda s: ', '.join(sorted(set(s)))),
            model_versions=('model_version', lambda s: ', '.join(sorted(set(s)))),
        )
        .sort_values(['used_for_targets', 'feature'])
        .reset_index(drop=True)
    )

    statline_table = pd.DataFrame(replay_rows).sort_values('target').reset_index(drop=True)

    display(Markdown('## Historical Replay Output'))
    display(Markdown('This output replays a real held-out game using only the pregame row that existed before the game was played.'))
    _display_styled_table('Game Context', context_table)
    _display_styled_table('Pregame Features Used By The Model', feature_table)
    _display_styled_table('Predicted vs Actual Statline', statline_table, {
        'prediction': '{:.2f}',
        'actual': '{:.0f}',
        'absolute_error': '{:.2f}',
    })

    return {
        'context': context_table,
        'features': feature_table,
        'statline': statline_table,
    }

available_eval_players = sorted(eval_df['player_name'].dropna().unique().tolist())
print(f'Available held-out players: {len(available_eval_players):,}')
print('Examples:', ', '.join(available_eval_players[:15]))

Available held-out players: 569
Examples: A.J. Lawson, AJ Green, AJ Johnson, Aaron Gordon, Aaron Holiday, Aaron Nesmith, Aaron Wiggins, Adam Flagler, Adama Sanogo, Adem Bona, Ajay Mitchell, Al Horford, Alec Burks, Alex Caruso, Alex Ducas


## Example Usage

The two cells below show the intended presentation flow:

1. List available held-out games for a recognizable player.
2. Replay one real game from the `2024-25` evaluation season.

In [6]:
list_player_games('Jayson Tatum').head(10)

,game_date,team,opponent,home_away,actual_points,actual_rebounds,actual_assists
15288,2024-10-22,BOS,NYK,home,37,4,10
15289,2024-10-24,BOS,WAS,away,25,11,6
15290,2024-10-26,BOS,DET,away,37,4,2
15291,2024-10-28,BOS,MIL,home,15,8,4
15292,2024-10-30,BOS,IND,away,37,8,4
15293,2024-11-01,BOS,CHA,away,32,11,3
15294,2024-11-02,BOS,CHA,away,29,7,3
15295,2024-11-04,BOS,ATL,away,28,6,9
15296,2024-11-06,BOS,GSW,home,32,4,2
15297,2024-11-08,BOS,BKN,home,33,9,6


In [7]:
historical_replay('Jayson Tatum', '2024-11-13')

## Historical Replay Output

This output replays a real held-out game using only the pregame row that existed before the game was played.

### Game Context

player_name,game_date,team,opponent,home_away,season,game_id
Jayson Tatum,2024-11-13,BOS,BKN,away,2024-25,0022400218


### Pregame Features Used By The Model

feature,value,used_for_targets,model_versions
l10_avg_usg,0.301000,"Assists, Points",V2
l5_avg_usg,0.301000,"Assists, Points",V2
missing_ast_l10,2.683333,"Assists, Points",V2
missing_ast_l5,2.983333,"Assists, Points",V2
missing_min_l10,41.532750,"Assists, Points",V2
missing_min_l5,43.024417,"Assists, Points",V2
missing_pts_l10,11.483333,"Assists, Points",V2
missing_pts_l5,12.883333,"Assists, Points",V2
missing_reb_l10,8.466667,"Assists, Points",V2
missing_reb_l5,9.266667,"Assists, Points",V2


### Predicted vs Actual Statline

target,model_version,model_family,prediction,actual,absolute_error
Assists,V2,LinearRegression,5.31,10,4.69
Points,V2,XGBRegressor,23.24,36,12.76
Rebounds,V0,LinearRegression,7.04,9,1.96


{'context':     player_name   game_date team opponent home_away   season     game_id
 0  Jayson Tatum  2024-11-13  BOS      BKN      away  2024-25  0022400218,
 'features':                 feature      value           used_for_targets model_versions
 0           l10_avg_usg      0.301            Assists, Points             V2
 1            l5_avg_usg      0.301            Assists, Points             V2
 2       missing_ast_l10   2.683333            Assists, Points             V2
 3        missing_ast_l5   2.983333            Assists, Points             V2
 4       missing_min_l10   41.53275            Assists, Points             V2
 5        missing_min_l5  43.024417            Assists, Points             V2
 6       missing_pts_l10  11.483333            Assists, Points             V2
 7        missing_pts_l5  12.883333            Assists, Points             V2
 8       missing_reb_l10   8.466667            Assists, Points             V2
 9        missing_reb_l5   9.266667            A

In [9]:
def find_demo_games(player_names, n_per_player=3):
    cols = [
        "player_name", "game_date", "team_abbr", "opponent_team", "home_away",
        "target_pts", "target_reb", "target_ast"
    ]
    optional_cols = ["teammates_out_count", "season_avg_min", "l5_avg_pts"]

    available_cols = [c for c in cols if c in eval_df.columns]
    available_optional = [c for c in optional_cols if c in eval_df.columns]

    out = []

    for player in player_names:
        temp = eval_df[eval_df["player_name"].str.lower() == player.lower()].copy()
        if temp.empty:
            continue

        sort_cols = []
        ascending = []

        if "teammates_out_count" in temp.columns:
            sort_cols.append("teammates_out_count")
            ascending.append(True)

        if "season_avg_min" in temp.columns:
            sort_cols.append("season_avg_min")
            ascending.append(False)

        sort_cols.append("game_date")
        ascending.append(False)

        temp = temp.sort_values(sort_cols, ascending=ascending)
        temp = temp[available_cols + available_optional].head(n_per_player)
        out.append(temp)

    if not out:
        return pd.DataFrame()

    return pd.concat(out).reset_index(drop=True)

In [12]:
demo_players = [
    "Jayson Tatum",
    "Stephen Curry",
    "Anthony Edwards",
    "Nikola Jokić",
    "Luka Dončić"
]

candidate_games = find_demo_games(demo_players, n_per_player=3)
display(candidate_games)

,player_name,game_date,team_abbr,opponent_team,home_away,target_pts,target_reb,target_ast,teammates_out_count,season_avg_min,l5_avg_pts
0,Jayson Tatum,2024-10-22,BOS,NYK,home,37,4,10,0,NaN,NaN
1,Jayson Tatum,2024-10-24,BOS,WAS,away,25,11,6,1,30.300000,37.000000
2,Jayson Tatum,2025-01-20,BOS,GSW,away,22,9,7,2,36.502051,24.400000
3,Stephen Curry,2024-10-25,GSW,UTA,away,20,3,4,0,25.071667,17.000000
4,Stephen Curry,2024-10-23,GSW,POR,away,17,9,10,0,NaN,NaN
5,Stephen Curry,2024-10-27,GSW,LAC,home,18,4,6,1,26.244167,18.500000
6,Anthony Edwards,2024-10-24,MIN,SAC,away,32,7,4,0,41.066667,27.000000
7,Anthony Edwards,2024-10-26,MIN,TOR,home,24,6,4,0,40.133333,29.500000
8,Anthony Edwards,2024-10-29,MIN,DAL,home,37,6,3,0,39.816667,27.666667
9,Nikola Jokić,2024-10-24,DEN,OKC,home,16,12,13,0,NaN,NaN


In [14]:
demo_examples = [
    ("Jayson Tatum", "2024-10-24"),
    ("Nikola Jokić", "2024-11-02"),
    ("Anthony Edwards", "2024-10-26"),
]

for player, game_date in demo_examples:
    print("=" * 80)
    print(f"Historical replay for {player} on {game_date}")
    historical_replay(player, game_date)

Historical replay for Jayson Tatum on 2024-10-24


## Historical Replay Output

This output replays a real held-out game using only the pregame row that existed before the game was played.

### Game Context

player_name,game_date,team,opponent,home_away,season,game_id
Jayson Tatum,2024-10-24,BOS,WAS,away,2024-25,0022400073


...

player_name,game_date,team,opponent,home_away,season,game_id
Anthony Edwards,2024-10-26,MIN,TOR,home,2024-25,0022400093


### Pregame Features Used By The Model

feature,value,used_for_targets,model_versions
l10_avg_usg,0.307000,"Assists, Points",V2
l5_avg_usg,0.307000,"Assists, Points",V2
missing_ast_l10,0.000000,"Assists, Points",V2
missing_ast_l5,0.000000,"Assists, Points",V2
missing_min_l10,0.000000,"Assists, Points",V2
missing_min_l5,0.000000,"Assists, Points",V2
missing_pts_l10,0.000000,"Assists, Points",V2
missing_pts_l5,0.000000,"Assists, Points",V2
missing_reb_l10,0.000000,"Assists, Points",V2
missing_reb_l5,0.000000,"Assists, Points",V2


### Predicted vs Actual Statline

target,model_version,model_family,prediction,actual,absolute_error
Assists,V2,LinearRegression,3.76,4,0.24
Points,V2,XGBRegressor,23.47,24,0.53
Rebounds,V0,LinearRegression,6.10,6,0.10
